# T-MG-N-SUPP Time-Domain Multifrequency Grouped Sensor-Noise Support-Recovery Baseline

Sensor-noise grouped support recovery with time-domain broadband-noise simulations. Signals are generated with `quick_sim(mode="noise")`, then frequency bins at or above 100 Hz are retained. Group LASSO and Sparse Prior use frequency grouping.


In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/cs_priors")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/cs_priors_matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn.metrics import f1_score

from cs_priors.simulation.mixing_model import quick_sim
from cs_priors.solvers.freq_lasso import frequency_lasso_solve
from cs_priors.solvers.freq_group_lasso import frequency_group_lasso_solve
from cs_priors.solvers.freq_sparse_prior import sparse_prior_solve


In [ ]:
TAG = "T-MG-N-SUPP"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / TAG
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, stem: str):
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", dpi=300, bbox_inches="tight")


SNR_DB = np.array([10, 15, 20, 30, 40, 50, 60])
SEEDS = np.arange(5)
NUM_MICS_FIXED = 6
MIC_COUNTS = np.array([6])

NUM_SOURCES = 10
NUM_ACTIVE = 2
MIN_FREQ_HZ = 100.0
SAMPLING_RATE = 2000.0
DURATION = 0.05
AMPLITUDE = 1.0

MIC_RADIUS = 0.3
SOURCE_DISTANCE = 1.5

LASSO_ALPHA = 1e-4
GROUP_LASSO_ALPHA = 1e-4
LASSO_MAX_ITER = 5_000
GROUP_LASSO_MAX_ITER = 500
THRESHOLD_FRACTION = 0.10
PLOT_FLOOR = 1e-16
GROUPING = "frequency"

METHOD_ORDER = ["r-LASSO", "Group LASSO", "Sparse Prior", "X_pinv"]
METHOD_COLORS = {
    "X_pinv": "tab:blue",
    "r-LASSO": "tab:orange",
    "Group LASSO": "tab:green",
    "Sparse Prior": "tab:red",
}
METHOD_LINESTYLES = {
    "X_pinv": ":",
    "r-LASSO": "-",
    "Group LASSO": "-",
    "Sparse Prior": "-",
}


In [ ]:
def make_simulation(num_mics: int, snr_db: float, seed: int):
    return quick_sim(
        num_mics=int(num_mics),
        num_sources=NUM_SOURCES,
        num_active=NUM_ACTIVE,
        seed=int(seed),
        sampling_rate=SAMPLING_RATE,
        duration=DURATION,
        source_distance=SOURCE_DISTANCE,
        mic_radius=MIC_RADIUS,
        array_type="circular",
        mic_angle_start=0.0,
        mic_angle_span=2 * np.pi,
        source_angle_start=0.0,
        source_angle_span=2 * np.pi,
        mode="noise",
        amplitude=AMPLITUDE,
        min_freq_hz=MIN_FREQ_HZ,
        single_frequency_hz=None,
        sensor_snr_db=float(snr_db),
        model_snr_db=None,
        inverse_method="mp",
    )


def solve_methods(sim) -> dict[str, np.ndarray]:
    X_pinv = sim.X_pinv
    return {
        "X_pinv": X_pinv,
        "r-LASSO": frequency_lasso_solve(
            sim.Y, sim.A, alpha=LASSO_ALPHA, max_iter=LASSO_MAX_ITER, X_start=X_pinv
        ),
        "Group LASSO": frequency_group_lasso_solve(
            sim.Y,
            sim.A,
            alpha=GROUP_LASSO_ALPHA,
            grouping=GROUPING,
            max_iter=GROUP_LASSO_MAX_ITER,
            X_start=X_pinv,
        ),
        "Sparse Prior": sparse_prior_solve(X_pinv, sim.A, grouping=GROUPING),
    }


def support_labels(sim) -> np.ndarray:
    labels = np.zeros(NUM_SOURCES, dtype=int)
    labels[np.asarray(sim.active_indices, dtype=int)] = 1
    return labels


def source_scores(X_hat: np.ndarray) -> np.ndarray:
    return np.linalg.norm(X_hat, axis=1)


def threshold_predictions(scores: np.ndarray) -> np.ndarray:
    max_score = float(np.max(scores))
    if max_score <= 0.0:
        return np.zeros_like(scores, dtype=int)
    return (scores >= THRESHOLD_FRACTION * max_score).astype(int)


def mean_squared_difference(X_hat: np.ndarray, X_true: np.ndarray) -> float:
    return float(np.mean(np.abs(X_hat - X_true) ** 2))


def assert_simulation_contract(sim, num_mics: int):
    assert sim.A.shape == (int(num_mics), NUM_SOURCES, 45)
    assert np.all(sim.freqs >= MIN_FREQ_HZ)
    assert len(sim.active_indices) == NUM_ACTIVE
    assert np.allclose(sim.delta, 0.0)
    assert np.linalg.norm(sim.eta) > 0.0


def run_case(num_mics: int, snr_db: float, seed: int) -> list[dict]:
    sim = make_simulation(num_mics, snr_db, seed)
    assert_simulation_contract(sim, num_mics)

    y_true = support_labels(sim)
    rows = []

    for method, X_hat in solve_methods(sim).items():
        scores = source_scores(X_hat)
        y_pred = threshold_predictions(scores)
        rows.append(
            {
                "tag": TAG,
                "num_mics": int(num_mics),
                "sensor_snr_db": float(snr_db),
                "seed": int(seed),
                "method": method,
                "f1_threshold_10_percent": f1_score(y_true, y_pred, zero_division=0),
                "mean_squared_difference": mean_squared_difference(X_hat, sim.X),
            }
        )

    return rows


In [ ]:
from itertools import product
from tqdm.auto import tqdm

rows = []
cases = product(MIC_COUNTS, SNR_DB, SEEDS)
total = len(MIC_COUNTS) * len(SNR_DB) * len(SEEDS)

for num_mics, snr_db, seed in tqdm(cases, total=total, desc="Running simulations"):
    rows.extend(run_case(int(num_mics), float(snr_db), int(seed)))

results_df = pd.DataFrame(rows)
results_df["method"] = pd.Categorical(results_df["method"], categories=METHOD_ORDER, ordered=True)
results_df = results_df.sort_values(["num_mics", "sensor_snr_db", "seed", "method"]).reset_index(drop=True)
results_df.head()


In [ ]:
summary_df = (
    results_df
    .groupby(["num_mics", "sensor_snr_db", "method"], observed=False)
    .agg(
        median_f1_threshold_10_percent=("f1_threshold_10_percent", "median"),
        q25_f1_threshold_10_percent=("f1_threshold_10_percent", lambda x: x.quantile(0.25)),
        q75_f1_threshold_10_percent=("f1_threshold_10_percent", lambda x: x.quantile(0.75)),
        median_mean_squared_difference=("mean_squared_difference", "median"),
        q25_mean_squared_difference=("mean_squared_difference", lambda x: x.quantile(0.25)),
        q75_mean_squared_difference=("mean_squared_difference", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)
summary_df.head()


In [ ]:
def plot_snr_metric(metric_stem: str, ylabel: str, title: str, yscale="linear", ylim=None):
    plot_df = summary_df[summary_df["num_mics"] == NUM_MICS_FIXED]
    fig, ax = plt.subplots(figsize=(7.0, 4.4))

    for method in METHOD_ORDER:
        method_df = plot_df[plot_df["method"] == method].sort_values("sensor_snr_db")
        x = method_df["sensor_snr_db"].to_numpy(dtype=float)
        median = method_df[f"median_{metric_stem}"].to_numpy(dtype=float)
        q25 = method_df[f"q25_{metric_stem}"].to_numpy(dtype=float)
        q75 = method_df[f"q75_{metric_stem}"].to_numpy(dtype=float)

        if yscale == "log":
            median = np.maximum(median, PLOT_FLOOR)
            q25 = np.maximum(q25, PLOT_FLOOR)
            q75 = np.maximum(q75, PLOT_FLOOR)

        ax.plot(
            x, median, marker="o", label=method,
            color=METHOD_COLORS[method], linestyle=METHOD_LINESTYLES[method]
        )
        ax.fill_between(x, q25, q75, color=METHOD_COLORS[method], alpha=0.18, linewidth=0)

    ax.set_xticks(SNR_DB)
    ax.set_xlabel("Sensor SNR (dB)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_yscale(yscale)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(True, which="both", linewidth=0.5, alpha=0.35)
    ax.legend(frameon=False)
    return fig, ax


In [ ]:
fig, ax = plot_snr_metric(
    "mean_squared_difference",
    "Mean squared difference",
    f"Coefficient MSE vs. sensor SNR (M={NUM_MICS_FIXED})",
)
save_figure(fig, "mean_squared_difference_vs_sensor_snr")
plt.show()


In [ ]:
fig, ax = plot_snr_metric(
    "f1_threshold_10_percent",
    "F1-score",
    f"Support F1 vs. sensor SNR (M={NUM_MICS_FIXED})",
    ylim=(-0.03, 1.03),
)
save_figure(fig, "f1_threshold_10_percent_vs_sensor_snr")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.4), sharex=True, sharey=True)
axes = axes.ravel()

for ax, method in zip(axes, METHOD_ORDER):
    heat_df = summary_df[summary_df["method"] == method]
    heat = (
        heat_df.pivot(index="num_mics", columns="sensor_snr_db", values="median_f1_threshold_10_percent")
        .reindex(index=MIC_COUNTS, columns=SNR_DB)
    )
    im = ax.imshow(heat.to_numpy(), origin="lower", aspect="auto", vmin=0.0, vmax=1.0)
    ax.set_title(method)
    ax.set_xticks(np.arange(len(SNR_DB)))
    ax.set_xticklabels([str(int(x)) for x in SNR_DB], rotation=45)
    ax.set_yticks(np.arange(len(MIC_COUNTS)))
    ax.set_yticklabels([str(int(x)) for x in MIC_COUNTS])

for ax in axes[2:]:
    ax.set_xlabel("Sensor SNR (dB)")
for ax in axes[::2]:
    ax.set_ylabel("Number of microphones")

fig.colorbar(im, ax=axes, label="Median F1-score")
fig.suptitle("Support F1 heatmap over sensor SNR and microphones", y=0.98)
save_figure(fig, "f1_threshold_10_percent_heatmap_sensor_snr_num_mics")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.4), sharex=True, sharey=True)
axes = axes.ravel()

heat_values = summary_df["median_mean_squared_difference"].to_numpy(dtype=float)
vmin = max(np.nanmin(heat_values), PLOT_FLOOR)
vmax = np.nanmax(heat_values)
if vmax <= vmin:
    vmax = vmin * 10.0

for ax, method in zip(axes, METHOD_ORDER):
    heat_df = summary_df[summary_df["method"] == method]
    heat = (
        heat_df.pivot(
            index="num_mics",
            columns="sensor_snr_db",
            values="median_mean_squared_difference",
        )
        .reindex(index=MIC_COUNTS, columns=SNR_DB)
    )

    im = ax.imshow(
        np.maximum(heat.to_numpy(dtype=float), PLOT_FLOOR),
        origin="lower",
        aspect="auto",
        norm=LogNorm(vmin=vmin, vmax=vmax),
    )
    ax.set_title(method)
    ax.set_xticks(np.arange(len(SNR_DB)))
    ax.set_xticklabels([str(int(x)) for x in SNR_DB], rotation=45)
    ax.set_yticks(np.arange(len(MIC_COUNTS)))
    ax.set_yticklabels([str(int(x)) for x in MIC_COUNTS])

for ax in axes[2:]:
    ax.set_xlabel("Sensor SNR (dB)")
for ax in axes[::2]:
    ax.set_ylabel("Number of microphones")

fig.colorbar(im, ax=axes, label="Median MSE")
fig.suptitle("Coefficient MSE heatmap over sensor SNR and microphones", y=0.98)
save_figure(fig, "mean_squared_difference_heatmap_sensor_snr_num_mics")
plt.show()


## Interpretation

This notebook tests sensor-noise support recovery after time-domain broadband-noise generation. Higher SNR means weaker sensor noise. This notebook differs from T-MI-N-SUPP only by enabling frequency grouping in Group LASSO and Sparse Prior. The F1 metric measures whether source row norms identify the active support, while MSE measures coefficient reconstruction accuracy.
